# 08_逐帧CNN与时间平均：参考答案

安排：第08课，接在07课之后、09课之前。目标：跟踪从视频到逐帧分类再到时间平均的shape，理解平均会丢弃顺序。

来源：Word手册的CNN/表示学习概念；飞书《A 观行知意 学习指导》关于逐帧CNN与时间聚合的介绍。飞书指出其Kaggle示例使用热成像；这里读取已有深度/IR，只借鉴思路，不导入热成像数据或权重。完整对照见README.md。

本示例没有训练，卷积和分类头为随机参数。可以运行并观察形状，不能作为比赛模型或效果证据。


In [ ]:
from pathlib import Path
import sys
sys.dont_write_bytecode = True
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'tools/learning.py').is_file() and (p / 'config.json').is_file())
sys.path.insert(0, str(ROOT / 'tools'))
import learning as L
OUT = ROOT / 'outputs'
OUT.mkdir(exist_ok=True)


## 空间特征与片段分类

同一个2D CNN处理每帧；分类头每帧返回40个logits。本例在类别维softmax，然后沿时间平均。它是明确选择的一种教学实现，未声称与Kaggle原代码逐行一致。


In [ ]:
import torch
from torch import nn
torch.set_num_threads(4)
torch.manual_seed(42)

class FrameMeanClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Conv2d(4, 8, kernel_size=3, padding=1),
            nn.ReLU(), nn.AdaptiveAvgPool2d(1), nn.Flatten())
        self.head = nn.Linear(8, 40)

    def forward(self, video):
        # video：[B,C,T,H,W]。先把时间移到通道前，再合并B与T。
        b, c, t, h, w = video.shape
        frames = video.permute(0, 2, 1, 3, 4).reshape(b*t, c, h, w)
        features = self.encoder(frames)         # [B*T,8]
        logits = self.head(features).reshape(b, t, 40)
        probabilities = logits.softmax(dim=-1) # 每帧40类分数
        return probabilities.mean(dim=1), logits

model = FrameMeanClassifier().eval()


## 真实片段经过教学网络

4通道来自伪彩深度3通道加IR1通道。permutation交换轴与reshape合并维度分开做；不能用reshape直接代替轴交换。


In [ ]:
clip, true_label, classes = L.load_clip('validation', 0, frames=16, size=128)
video = torch.from_numpy(clip).unsqueeze(0)
with torch.inference_mode():
    clip_probs, frame_logits = model(video)
assert video.shape == (1, 4, 16, 128, 128)
assert frame_logits.shape == (1, 16, 40)
assert clip_probs.shape == (1, 40)
torch.testing.assert_close(clip_probs.sum(dim=-1), torch.ones(1))
print('输入BCTHW：', tuple(video.shape))
print('每帧分数：', tuple(frame_logits.shape))
print('聚合结果：', tuple(clip_probs.shape))
print('只检查形状；随机教学模型尚未学会识别动作，不报告其准确率。')


## 时间索引练习答案


In [ ]:
permutation = torch.arange(video.shape[2] - 1, -1, -1)
with torch.inference_mode():
    your_answer = model(video[:, :, permutation, :, :])[0]
torch.testing.assert_close(your_answer, clip_probs, atol=1e-6, rtol=1e-5)
assert your_answer.shape == (1, 40)
print('练习通过：打乱同一组帧不会改变这个简单聚合器的结果。')


## 解释

时间维是第2轴，选择video[:, :, permutation, :, :]改变的是帧序。每帧独立处理后做平均，所以顺序变化不会改变结果。这个结论只针对本教学结构，不能直接推广到具有时间卷积、状态或位置编码的模型。
